In [1]:
!git clone https://github.com/trhuyyy13/prompt-tuning-api-migration /kaggle/working/repo

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 54 (delta 16), reused 52 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 5.68 MiB | 4.96 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [2]:
!pip install -q -r /kaggle/working/repo/prompt_tuning_deepseek/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.6 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Baseline: model gốc codegen-2B-mono (chưa prompt-tune)
# Kết quả này dùng để so sánh với sau khi train.
!python /kaggle/working/repo/prompt_tuning_deepseek/test_forget_quality.py   --model_name_or_path Salesforce/codegen-2B-mono   --baseline   --data_file /kaggle/working/repo/data_raw/outdated_y+_FINAL.json   --limit 200   --batch_size 4   --output_dir /kaggle/working/outputs/prompt_tuning_codegen2b_baseline

In [ ]:
# Baseline: model gốc deepseek-coder-1.3b-base (chưa prompt-tune)
!python /kaggle/working/repo/prompt_tuning_deepseek/test_forget_quality.py   --model_name_or_path deepseek-ai/deepseek-coder-1.3b-base   --baseline   --data_file /kaggle/working/repo/data_raw/outdated_y+_FINAL.json   --limit 200   --batch_size 8   --output_dir /kaggle/working/outputs/prompt_tuning_deepseek_baseline

In [ ]:
!python /kaggle/working/repo/prompt_tuning_deepseek/train_prompt_tuning.py \
  --model_name_or_path Salesforce/codegen-2B-mono \
  --train_file /kaggle/working/repo/data_raw/outdated_y+_FINAL.json \
  --valid_file /kaggle/working/repo/data_raw/outdated_y+_FINAL.json \
  --output_dir /kaggle/working/outputs/prompt_tuning_codegen2b_global \
  --num_virtual_tokens 20 \
  --prompt_init random \
  --max_input_length 512 \
  --max_target_length 128 \
  --max_seq_length 640 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --learning_rate 5e-3 \
  --num_train_epochs 1 \
  --warmup_ratio 0.03 \
  --logging_steps 10 \
  --eval_steps 200 \
  --save_steps 200 \
  --seed 42 \
  --fp16

In [ ]:
!python /kaggle/working/repo/prompt_tuning_deepseek/push_to_hub.py \
  --checkpoint_dir /kaggle/working/outputs/prompt_tuning_codegen2b_global \
  --repo_id HuyTran1301/depapi-soft-prompt-codegen2b-test \
  --commit_message "soft prompt codegen-2B-mono"

In [ ]:
!python /kaggle/working/repo/prompt_tuning_deepseek/test_forget_quality.py   --model_name_or_path Salesforce/codegen-2B-mono   --checkpoint_dir /kaggle/working/outputs/prompt_tuning_codegen2b_global   --data_file /kaggle/working/repo/data_raw/outdated_y+_FINAL.json   --limit 20   --batch_size 4   --output_dir /kaggle/working/outputs/prompt_tuning_codegen2b_global2